# Lab 1 (LangChain build) · Simple RAG, made visible — free Kaggle T4

**The proof.** This is the *exact same lab* as `lab1_simple_rag_kaggle.ipynb` — same
k8s Q&A dataset, same `bge-small` embedder, same FAISS index, same Qwen2.5-3B answer —
but **rebuilt with [LangChain](https://python.langchain.com)** instead of by hand. Read
the two side by side and you'll see the point from the folder README:

> LangChain doesn't add a new concept to this layer — it **productizes the seven
> phases** you built by hand. Each phase becomes one pre-built, swappable component.

We keep the lab *made visible*: even through LangChain we still print the chunks a
query loads, grow top-k, and compare chunk sizes — because the phases are still there,
just behind a tidier API. The final mapping table is the receipt.

**Settings:** Accelerator = **GPU T4 x2** (not P100), Internet = **On**, then **Run All**.
Only the final answer cell uses the GPU.

*Data: [`ItshMoh/kubernetes_qa_pairs`](https://huggingface.co/datasets/ItshMoh/kubernetes_qa_pairs).*

In [ ]:
import warnings, logging, os
warnings.filterwarnings('ignore')
os.environ['TRANSFORMERS_VERBOSITY'] = 'error'
os.environ['HF_HUB_VERBOSITY'] = 'error'
os.environ['PYDEVD_DISABLE_FILE_VALIDATION'] = '1'
for _n in ('huggingface_hub','sentence_transformers','transformers','datasets','langchain'):
    logging.getLogger(_n).setLevel(logging.ERROR)

# the LangChain stack (split into focused packages) + the same engines underneath
!pip install -q langchain langchain-community langchain-huggingface langchain-text-splitters \
               faiss-cpu sentence-transformers datasets 2>/dev/null

import textwrap

## 1 · A real public dataset (Phase 1 · Sources)

Same source as the hand-built lab — ~500 real k8s Q&A pairs. The only LangChain-ism
here is wrapping each answer in a **`Document`** (page content + metadata), the unit
every LangChain loader/splitter/retriever speaks. The `metadata` dict is exactly the
Phase-2 metadata (id, source, tags) the README talks about.

In [ ]:
from datasets import load_dataset
from langchain_core.documents import Document

ds = load_dataset('ItshMoh/kubernetes_qa_pairs', split='train')

docs = [Document(page_content=r['answer'],
                 metadata={'id': f'qa-{i}', 'question': r['question'], 'topic': r['topic']})
        for i, r in enumerate(ds) if (r['answer'] or '').strip()]

print('rows:', len(ds), '| Documents built:', len(docs))
print('example metadata:', docs[0].metadata)

## 2 · Embed + index (Phases 3-4) — *one call each*

This is where the hand-built lab spent the most code: load `SentenceTransformer`,
`.encode(...)` with `normalize_embeddings`, build a `faiss.IndexFlatIP`, `.add(...)`,
and hand-write a `retrieve()` that re-embeds the query with the right prefix.

LangChain collapses all of it:

- **`HuggingFaceBgeEmbeddings`** = Phase 3. Wraps the *same* `bge-small`, and knows
  bge's query prefix (`Represent this sentence...`) so you never hand-prepend it.
- **`FAISS.from_documents`** = Phase 4. Embeds every doc and builds the index in one
  line. We pass `MAX_INNER_PRODUCT` + normalized vectors to match the hand-built
  `IndexFlatIP` cosine search exactly.
- **`.as_retriever()`** = Phase 5. Hands back a `retrieve()` for free.

In [ ]:
from langchain_community.embeddings import HuggingFaceBgeEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.vectorstores.utils import DistanceStrategy

emb = HuggingFaceBgeEmbeddings(                       # Phase 3 — same bge-small, on CPU
    model_name='BAAI/bge-small-en-v1.5',
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True},
    query_instruction='Represent this sentence for searching relevant passages: ')

store = FAISS.from_documents(                         # Phase 4 — embed + index in ONE call
    docs, emb, distance_strategy=DistanceStrategy.MAX_INNER_PRODUCT)

retriever = store.as_retriever(search_kwargs={'k': 3})   # Phase 5 — top-k retriever
print('indexed', store.index.ntotal, 'answer-chunks')

## 3 · See which chunks get loaded

The whole game, unchanged: retrieval returns the nearest `Document`s, and **those —
and only those — go into the prompt.** `similarity_search_with_score` is LangChain's
window into Phase 5; print them and you can debug RAG the same way as the hand-built
lab (most failures are *retrieval* failures).

In [ ]:
def show_hits(query, k=3):
    print(f'QUERY: {query}\n')
    print(f'--- top-{k} chunks loaded into the prompt ---')
    for rank, (d, score) in enumerate(store.similarity_search_with_score(query, k=k), 1):
        print(f"#{rank}  score={score:.3f}  topic={d.metadata['topic']}")
        print('    Q: ' + d.metadata['question'])
        print('    A: ' + textwrap.fill(d.page_content, 84).replace('\n', '\n       '))

show_hits('What does the kube-scheduler consider when selecting a node for a Pod?', k=3)

## 4 · top-k — how many chunks to load

Same dial as before. In LangChain it's just `search_kwargs={'k': k}` — no re-wiring.
- too small (k=1): one slightly-off chunk and the model has nothing to lean on
- too big (k=10): the real answer gets buried in noise, and you pay for the tokens

In [ ]:
q = 'What does the kube-scheduler consider when selecting a node for a Pod?'
for k in (1, 3, 6):
    hits = store.similarity_search_with_score(q, k=k)
    print(f'k={k}: ' + ' | '.join(f"{d.metadata['topic']}({s:.2f})" for d, s in hits))

## 5 · Chunk size — how big each piece is (Phase 2)

We indexed each answer whole; real docs must be **split** first. The hand-built lab
wrote its own word-splitter — LangChain ships **`RecursiveCharacterTextSplitter`**
(the README's *recursive* strategy: split on paragraphs → sentences → words until it
fits). Same trade-off, same result:
- **small** chunks → pinpoint retrieval, but a fact split across two is lost
- **large** chunks → full context, but the matching signal gets diluted

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
import pandas as pd

df = pd.DataFrame({'answer': [d.page_content for d in docs],
                   'topic':  [d.metadata['topic'] for d in docs]})
topic = df['topic'].value_counts().index[0]
long_text = ' '.join(df[df['topic'] == topic]['answer'].tolist())
print(f'topic "{topic}": {len(long_text.split())} words of source text\n')

def mini_store(text, chunk_size, overlap):
    splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=overlap)
    pieces = splitter.split_text(text)
    return pieces, FAISS.from_texts(pieces, emb, distance_strategy=DistanceStrategy.MAX_INNER_PRODUCT)

small_cs, small_ix = mini_store(long_text, chunk_size=140, overlap=28)   # ~small
big_cs,   big_ix   = mini_store(long_text, chunk_size=560, overlap=112)  # ~large
print(f'same text -> {len(small_cs)} small chunks  vs  {len(big_cs)} big chunks\n')

query = f'Tell me about {topic.lower()} in Kubernetes'
print('SMALL chunk retrieved (precise, may lack context):')
print('  ' + textwrap.fill(small_ix.similarity_search(query, k=1)[0].page_content, 86).replace('\n', '\n  '))
print('\nBIG chunk retrieved (more context, less precise):')
print('  ' + textwrap.fill(big_ix.similarity_search(query, k=1)[0].page_content, 86).replace('\n', '\n  '))

## 6 · Feed the chunks to the model → grounded, cited answer (Phase 6)

The hand-built lab manually applied the chat template, tokenized, called `.generate`,
and sliced the output. LangChain expresses the same flow as a **chain (LCEL)**:

```
retriever → format chunks into CONTEXT → prompt → LLM → text
```

- **`HuggingFacePipeline` / `ChatHuggingFace`** wrap the *same* Qwen2.5-3B and apply
  its chat template for us.
- **`ChatPromptTemplate`** holds the same open-book system rule (answer only from
  CONTEXT, cite ids, else *I don't know*).
- The `|` pipe is the chain. `rag_chain.invoke(question)` runs all of Phases 5-6.

(This cell loads the LLM — ~6 GB on the T4.)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# GPU-LLM cell, guarded: Kaggle's P100 can't run prebuilt fp16 kernels ('no kernel
# image'). Pick Accelerator = GPU T4 x2 to get the grounded answer; on P100 the CPU
# phases above still prove the LangChain mapping.
try:
    MODEL_ID = 'Qwen/Qwen2.5-3B-Instruct'
    tok = AutoTokenizer.from_pretrained(MODEL_ID)
    mdl = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float16, device_map='auto').eval()
    pipe = pipeline('text-generation', model=mdl, tokenizer=tok,
                    max_new_tokens=200, do_sample=False, return_full_text=False)
    chat = ChatHuggingFace(llm=HuggingFacePipeline(pipeline=pipe))

    SYS = ('You are a Kubernetes assistant. Use ONLY the CONTEXT to answer, and cite the chunk ids in [brackets]. '
           "Do not use any outside knowledge. If the answer is not in the CONTEXT, reply exactly: I don't know.")
    prompt = ChatPromptTemplate.from_messages([
        ('system', SYS),
        ('human', 'CONTEXT:\n{context}\n\nQUESTION: {question}')])

    def format_docs(ds):
        return '\n'.join(f"[{d.metadata['id']}] {d.page_content}" for d in ds)

    rag_chain = ({'context': retriever | format_docs, 'question': RunnablePassthrough()}
                 | prompt | chat | StrOutputParser())

    print('ANSWER:\n', textwrap.fill(
        rag_chain.invoke('What is Role-Based Access Control (RBAC) in Kubernetes?'), 90))
    # refusal: nothing in a k8s corpus answers this
    print('\nREFUSAL (out-of-corpus question):\n', textwrap.fill(
        rag_chain.invoke('What is the capital of France?'), 90))
except Exception as e:
    print('LLM cell skipped:', type(e).__name__, '-', str(e)[:140])
    print('This is the Kaggle P100 limitation, not LangChain. Set Accelerator = GPU T4 x2 and Run All.')

## The receipt · same seven phases, one component each

| Phase (folder README) | Hand-built Lab 1 | This LangChain build |
|---|---|---|
| 1 · Sources | dicts of `{id, text, topic}` | `Document(page_content, metadata)` |
| 2 · Chunk | hand-written `word_chunks()` | `RecursiveCharacterTextSplitter` |
| 3 · Embeddings | `SentenceTransformer(...).encode(...)` | `HuggingFaceBgeEmbeddings` |
| 4 · Vector store | `faiss.IndexFlatIP` + `.add` | `FAISS.from_documents(...)` |
| 5 · Retrieval | hand-written `retrieve(q, k)` | `store.as_retriever(k=...)` |
| 6 · RAG prompt | manual `apply_chat_template` + `.generate` | `ChatPromptTemplate \| ChatHuggingFace` (LCEL) |

### Takeaways

- **It's the same pipeline.** Every cell here has a one-to-one twin in the hand-built
  lab. LangChain added *no new concept* — it named each phase and made it swappable
  (change `FAISS` → `Chroma`, or `bge` → OpenAI, by editing one line).
- **The trade-off is visibility.** `rag_chain.invoke(q)` is great for *shipping* but
  hides Phases 5-6; that's exactly why the teaching lab builds them by hand first. You
  can only debug what you can see — so we still print the chunks (Section 3).
- **Still out of scope, framework or not:** Phase 7 evaluation (RAGAS / LangSmith) and
  Phase 8 data-ops. A framework writes the chain; it doesn't operate it.

Learn the phases here, and LangChain (or LlamaIndex) is just learning which method
name wraps each one. **Lab 2** adds the reranker and *measures* the lift.